In [15]:
import mygene
import pandas as pd

def get_mouse_human_mapping(mouse_gene_list):
    mg = mygene.MyGeneInfo()
    
    print("正在查询同源信息...")
    # scopes: 输入的类型（symbol）；fields: 需要获取的字段（人类同源基因 ensembl id）；target_species: 目标物种
    results = mg.querymany(mouse_gene_list, 
                           scopes='symbol', 
                           fields='homologene', 
                           species='mouse', 
                           as_dataframe=True)
    
    # 提取人类同源物的 Ensembl ID
    # 注意：mygene 的返回结构可能较复杂，需要根据结果清洗
    return results

# 示例：假设您已经通过 scanpy 读取了小鼠数据
mouse_genes = adata.var_names.tolist()
mapping_df = get_mouse_human_mapping(mouse_genes)

NameError: name 'adata' is not defined

In [ ]:
import pandas as pd
import os

# 1. 设置路径
base_path = '/public/home/202510191939/CellFM-demo'
# 确保路径连接正确
original_vocab_path = os.path.join(base_path, 'csv/expand_gene_info.csv')

# 2. 读取原始词表并获取基因列表
df_old = pd.read_csv(original_vocab_path)
# 修正列名为 'HGNC_gene'
symbols = df_old['HGNC_gene'].unique().tolist()
print(f"成功读取词表，共有 {len(symbols)} 个原始基因符号。")

# 3. 核心免疫基因映射备选方案
core_immune_map = {
    "CD4": "ENSG00000010610",
    "CD8A": "ENSG00000153563",
    "CD3E": "ENSG00000198851",
    "CD19": "ENSG00000177455",
    "PTPRC": "ENSG00000081237",
    "FOXP3": "ENSG00000049768",
    "ITGAM": "ENSG00000169896",
}

print("正在转换 ID...") 
try:
    import mygene
    mg = mygene.MyGeneInfo()
    # 仅针对词表中的基因进行查询
    results = mg.querymany(symbols, scopes='symbol', fields='ensembl.gene', species='human', as_dataframe=True)
    
    if 'ensembl.gene' in results.columns:
        mapping_df = results[['ensembl.gene']].reset_index()
        mapping_df.columns = ['symbol', 'ensembl_id']
        mapping_df = mapping_df.dropna()
    else:
        raise Exception("MyGene 结果中未发现 Ensembl ID 列")
except Exception as e:
    print(f"在线转换失败 ({e})，使用核心免疫映射表。")
    mapping_df = pd.DataFrame(list(core_immune_map.items()), columns=['symbol', 'ensembl_id'])

# 4. 保存映射表
map_output_path = os.path.join(base_path, 'human_map.csv')
mapping_df.to_csv(map_output_path, index=False)
print(f"映射表已保存至: {map_output_path}")

成功读取词表，共有 27855 个原始基因符号。
正在转换 ID...


Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
1962 input query terms found dup hits:	[('A2ML1-AS1', 2), ('A2ML1-AS2', 2), ('AADACL2-AS1', 3), ('ABHD15-AS1', 2), ('ABR-AS1', 3), ('ACAP2-
220 input query terms found no hit:	['ADAL', 'AKAP2', 'ARHGEF19-AS1', 'ARMT1', 'ATP6V1FNB', 'B3GNTL1', 'BMT2', 'BVES', 'BVES-AS1', 'C1or


映射表已保存至: /public/home/202510191939/CellFM-demo/human_map.csv


In [13]:
def build_final_cross_vocab(original_csv, mapping_csv, save_path):
    df_old = pd.read_csv(original_csv)
    mapping = pd.read_csv(mapping_csv)
    symbol_to_id = dict(zip(mapping['symbol'], mapping['ensembl_id']))
    
    # 按照原始顺序翻译（保持 27855 个索引不动）
    new_vocab = []
    for sym in df_old['HGNC_gene']:
        # 如果有 Ensembl ID 则转换，否则保留 Symbol 占位以维持权重对应关系
        new_vocab.append(symbol_to_id.get(sym, sym))
        
    old_size = len(new_vocab)
    
    # 根据 GeneCompass 论文方法，追加小鼠特有基因 ID [cite: 70, 71]
    # 这里您可以根据实际小鼠数据集中的非同源基因列表进行填充
    mouse_specific_ids = [] 
    
    final_vocab = new_vocab + mouse_specific_ids
    
    pd.DataFrame({'gene_id': final_vocab}).to_csv(save_path, index=False)
    print(f"跨物种词表构建完成！原始大小: {old_size}, 新大小: {len(final_vocab)}")
    return old_size, len(final_vocab)

# 执行构建
cross_vocab_path = os.path.join(base_path, 'cross_vocab.csv')
OLD_SIZE, NEW_SIZE = build_final_cross_vocab(original_vocab_path, map_output_path, cross_vocab_path)

跨物种词表构建完成！原始大小: 27855, 新大小: 27855


In [10]:
import pandas as pd
import os

base_path = '/public/home/202510191939/CellFM-demo'
# 确保使用修正后的路径连接方式
original_vocab_path = os.path.join(base_path, 'csv/expand_gene_info.csv')

# 读取文件
df_test = pd.read_csv(original_vocab_path)

# 打印前几行和所有列名，确认实际内容
print("文件中的实际列名有:", df_test.columns.tolist())
print(df_test.head())

文件中的实际列名有: ['HGNC_gene', 'h5_sample_frequency', 'biotype', 'gene_length', 'exon_count', 'exon_length_total', 'unique_exon_count', 'unique_exon_length_total', 'feature']
  HGNC_gene  h5_sample_frequency                    biotype  gene_length  \
0      A1BG                17850  gene with protein product       8315.0   
1  A1BG-AS1                16096       RNA, long non-coding          NaN   
2      A1CF                 7303  gene with protein product      86267.0   
3       A2M                16867  gene with protein product      48566.0   
4   A2M-AS1                15163       RNA, long non-coding          NaN   

   exon_count  exon_length_total  unique_exon_count  unique_exon_length_total  \
0        24.0             9209.0               22.0                    8639.0   
1         NaN                NaN                NaN                       NaN   
2       104.0            31017.0               32.0                   12489.0   
3        87.0            11592.0               81.

In [9]:
import pandas as pd
import os

# 1. 定义路径
base_path = '/public/home/202510191939/CellFM-demo/'
original_vocab_path = os.path.join(base_path, 'csv/expand_gene_info.csv')

# 2. 读取原始词表 
print(original_vocab_path)
df_old = pd.read_csv(original_vocab_path)
symbols = df_old['gene_name'].unique().tolist()

# 3. 核心免疫基因映射备选方案 (防止网络再次崩溃)
# 这里手动列出一些免疫专精模型最关键的基因映射
core_immune_map = {
    "CD4": "ENSG00000010610",
    "CD8A": "ENSG00000153563",
    "CD3E": "ENSG00000198851",
    "CD19": "ENSG00000177455",
    "PTPRC": "ENSG00000081237", # CD45
    "FOXP3": "ENSG00000049768",
    "ITGAM": "ENSG00000169896", # CD11b
}

print("正在尝试通过 mygene 转换 ID...")
try:
    import mygene
    mg = mygene.MyGeneInfo()
    # 批量查询
    results = mg.querymany(symbols, scopes='symbol', fields='ensembl.gene', species='human', as_dataframe=True)
    
    # 整理结果
    if 'ensembl.gene' in results.columns:
        mapping_df = results[['ensembl.gene']].reset_index()
        mapping_df.columns = ['symbol', 'ensembl_id']
        mapping_df = mapping_df.dropna()
    else:
        raise Exception("MyGene 返回结果中不含 ensembl ID")
        
except Exception as e:
    print(f"网络查询失败: {e}，正在使用核心免疫映射表生成基础文件...")
    mapping_df = pd.DataFrame(list(core_immune_map.items()), columns=['symbol', 'ensembl_id'])

# 4. 保存为 human_map.csv
output_path = os.path.join(base_path, 'human_map.csv')
mapping_df.to_csv(output_path, index=False)
print(f"文件已生成在: {output_path}")

/public/home/202510191939/CellFM-demo/csv/expand_gene_info.csv


KeyError: 'gene_name'